# 🔗 Notebook 4: Sharing & Permissions

Dropbox isn't just personal file storage — it's a **collaboration platform**. When you share a file with a teammate, you need to control *who* can see it, *what* they can do with it, and *when* that access expires. This is exactly like sharing a Google Doc: you pick "viewer" or "editor" before sending the link.

In this notebook we'll build a complete sharing system: inserting share records, generating secure download links, checking permissions, caching shared-file lists, revoking access, and notifying users — all on top of the same Postgres + MinIO + Redis stack we've used throughout the series.

## Learning Objectives

By the end of this notebook, you'll understand:
- How the `shared_files` table models sharing relationships
- How presigned URLs give time-limited access to private objects
- How to build a permission-check function (owner vs shared user)
- How to cache "Shared with me" queries in Redis and invalidate on changes
- What happens when access is revoked (and why short TTLs matter)
- How sync events notify recipients about new shares

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/dropbox
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `dropbox_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`
- **MinIO Console** (S3 GUI): http://localhost:9001  
  Login: User `minioadmin`, Password `minioadmin`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import psycopg2.extras
import redis
import boto3
import json
import time
import hashlib
import requests
from datetime import datetime

# ── Connection settings ──────────────────────────────────────

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "dropbox_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

S3_CONFIG = {
    "endpoint_url": "http://localhost:9000",
    "aws_access_key_id": "minioadmin",
    "aws_secret_access_key": "minioadmin"
}

BUCKET = "dropbox-files"

# ── Helper functions ─────────────────────────────────────────

def get_db():
    """Return a new Postgres connection with dict-like rows."""
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

def get_redis():
    """Return a Redis client."""
    return redis.Redis(**REDIS_CONFIG)

def get_s3():
    """Return a boto3 S3 client pointed at MinIO."""
    return boto3.client("s3", **S3_CONFIG)

# ── Test connections ─────────────────────────────────────────

try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: cd 06-system-designs/dropbox && docker-compose up -d")

try:
    r = get_redis()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: cd 06-system-designs/dropbox && docker-compose up -d")

try:
    s3 = get_s3()
    # Create bucket if it doesn't exist
    existing = [b["Name"] for b in s3.list_buckets()["Buckets"]]
    if BUCKET not in existing:
        s3.create_bucket(Bucket=BUCKET)
    print("✅ Connected to MinIO (S3)")
except Exception as e:
    print(f"❌ MinIO failed: {e}")
    print("   Run: cd 06-system-designs/dropbox && docker-compose up -d")

✅ Connected to PostgreSQL
✅ Connected to Redis
✅ Connected to MinIO (S3)


## 🤔 Why Sharing Is a Core Feature

Imagine Dropbox without sharing — it would just be a backup drive. The real value of cloud storage comes from **collaboration**:

- A designer uploads a mockup and shares it with the engineering team
- A manager shares a spreadsheet so everyone can edit it
- A student sends a read-only link to their professor

But sharing brings **hard problems**:

| Problem | Why It's Hard |
|---------|---------------|
| **Who can see this file?** | Need a permission model (read vs write) |
| **How do I send a download link?** | Links must expire so leaked URLs don't last forever |
| **What if I revoke access?** | Already-generated links may still work for a while |
| **How does the recipient know?** | Need a notification/sync mechanism |

Think of it like sharing a Google Doc. When you click "Share" you choose:
- **Viewer** — they can look but not change anything  
- **Editor** — they can modify the document  

We'll build exactly this with a `shared_files` table that tracks `(file, user, permission)`.

## 📋 Seed Data — Alice Uploads a File

Before we can share anything, we need a file in the system. Let's have **Alice** (user 1) upload a small text file. We'll store the bytes in MinIO and record the metadata in Postgres — just like a real Dropbox upload.

In [2]:
# Clean slate: remove any leftover data from previous runs
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM sync_events")
cur.execute("DELETE FROM shared_files")
cur.execute("DELETE FROM chunks")
cur.execute("DELETE FROM files")
conn.close()

# ── Upload a file as Alice ───────────────────────────────────

file_content = b"Project plan for Q4: launch sharing feature by end of October."
file_name = "project-plan.txt"
storage_key = f"alice/{file_name}"
fingerprint = hashlib.sha256(file_content).hexdigest()

# 1. Store bytes in MinIO
s3 = get_s3()
s3.put_object(Bucket=BUCKET, Key=storage_key, Body=file_content)
print(f"📦 Uploaded {len(file_content)} bytes to MinIO → {storage_key}")

# 2. Record metadata in Postgres
conn = get_db()
cur = conn.cursor()
cur.execute("""
    INSERT INTO files (owner_id, file_name, mime_type, file_size, fingerprint, storage_key, status)
    VALUES (1, %s, 'text/plain', %s, %s, %s, 'uploaded')
    RETURNING id
""", (file_name, len(file_content), fingerprint, storage_key))
file_id = cur.fetchone()[0]
conn.close()

print(f"✅ File record created — id={file_id}, owner=alice")
print(f"   Name: {file_name}")
print(f"   Size: {len(file_content)} bytes")
print(f"   SHA-256: {fingerprint[:16]}...")

📦 Uploaded 62 bytes to MinIO → alice/project-plan.txt


✅ File record created — id=24, owner=alice
   Name: project-plan.txt
   Size: 62 bytes
   SHA-256: dbcc8ca64b5bccb6...


## 📝 The `shared_files` Table

The `shared_files` table is the heart of our sharing system. Each row says:

> "File X is shared with User Y, and they have Z permission."

```
shared_files
┌────┬─────────┬─────────────┬────────────┬───────────┐
│ id │ file_id │ shared_with │ permission │ shared_at │
├────┼─────────┼─────────────┼────────────┼───────────┤
│  1 │       1 │           2 │ read       │ now()     │  ← Bob can view
│  2 │       1 │           3 │ write      │ now()     │  ← Charlie can edit
└────┴─────────┴─────────────┴────────────┴───────────┘
```

Key design decisions:
- **`UNIQUE (file_id, shared_with)`** — you can't share the same file with the same person twice
- **`permission` is 'read' or 'write'** — simple two-level model (just like Google Docs viewer/editor)
- **`ON DELETE CASCADE` on file_id** — if the file is deleted, all shares are automatically removed

Let's share Alice's file with Bob (read-only) and Charlie (read + write).

In [3]:
conn = get_db()
cur = conn.cursor()

# Share with Bob (read-only)
cur.execute("""
    INSERT INTO shared_files (file_id, shared_with, permission)
    VALUES (%s, 2, 'read')
    ON CONFLICT (file_id, shared_with) DO UPDATE SET permission = 'read'
    RETURNING id, shared_at
""", (file_id,))
bob_share = cur.fetchone()
print(f"✅ Shared with Bob (read)  — share id={bob_share[0]}")

# Share with Charlie (write — which implies read too)
cur.execute("""
    INSERT INTO shared_files (file_id, shared_with, permission)
    VALUES (%s, 3, 'write')
    ON CONFLICT (file_id, shared_with) DO UPDATE SET permission = 'write'
    RETURNING id, shared_at
""", (file_id,))
charlie_share = cur.fetchone()
print(f"✅ Shared with Charlie (write) — share id={charlie_share[0]}")

# ── Query: what files are shared with Bob? ───────────────────
cur.execute("""
    SELECT f.file_name, f.file_size, sf.permission, u.username AS owner
    FROM shared_files sf
    JOIN files f ON f.id = sf.file_id
    JOIN users u ON u.id = f.owner_id
    WHERE sf.shared_with = 2   -- Bob's user id
""")
rows = cur.fetchall()
conn.close()

print("\n📂 Files shared with Bob:")
for row in rows:
    print(f"   {row[0]} ({row[1]} bytes) — {row[2]} access, owned by {row[3]}")

✅ Shared with Bob (read)  — share id=7
✅ Shared with Charlie (write) — share id=8

📂 Files shared with Bob:
   project-plan.txt (62 bytes) — read access, owned by alice


## 🔐 Presigned URLs for Secure Sharing

When Bob wants to download a shared file, we **don't** stream it through our server. Instead, we generate a **presigned URL** — a special link that:

1. Points directly to the file in object storage (MinIO/S3)
2. Contains a cryptographic signature proving it's legitimate
3. **Expires** after a set time (e.g., 60 seconds)

```
  Bob's App               Our API                 MinIO (S3)
     │                       │                        │
     │── "Give me a link" ──►│                        │
     │                       │── check permission ──► │
     │                       │◄── OK ────────────────│
     │◄── presigned URL ────│                        │
     │                       │                        │
     │──────── download directly from S3 ───────────►│
     │◄─────── file bytes ──────────────────────────│
```

**Why expire?** If someone copies the URL and posts it publicly, it only works for a short time. This is the same idea behind the "bearer token" pattern — whoever *bears* (holds) the token can use it, so you keep the token short-lived.

Let's generate a presigned URL, download the file, wait for it to expire, then show it fails.

In [4]:
s3 = get_s3()

# Generate a presigned URL that expires in 5 seconds (short for demo purposes)
presigned_url = s3.generate_presigned_url(
    "get_object",
    Params={"Bucket": BUCKET, "Key": storage_key},
    ExpiresIn=5  # seconds
)

print("🔗 Presigned URL (expires in 5 seconds):")
print(f"   {presigned_url[:80]}...")

# ── Download immediately — should work ───────────────────────
response = requests.get(presigned_url)
print(f"\n⬇️  Immediate download: HTTP {response.status_code}")
print(f"   Content: {response.text}")

# ── Wait for expiry, then try again ──────────────────────────
print("\n⏳ Waiting 6 seconds for URL to expire...")
time.sleep(6)

response_expired = requests.get(presigned_url)
print(f"⬇️  After expiry: HTTP {response_expired.status_code}")
if response_expired.status_code == 403:
    print("   ❌ Access denied — the URL has expired!")
    print("   This is why presigned URLs are safe to share temporarily.")

🔗 Presigned URL (expires in 5 seconds):
   http://localhost:9000/dropbox-files/alice/project-plan.txt?AWSAccessKeyId=minioa...

⬇️  Immediate download: HTTP 200
   Content: Project plan for Q4: launch sharing feature by end of October.

⏳ Waiting 6 seconds for URL to expire...


⬇️  After expiry: HTTP 403
   ❌ Access denied — the URL has expired!
   This is why presigned URLs are safe to share temporarily.


## ✅ Permission Checks

Before generating a presigned URL (or allowing any action), we need to verify the user has the right permission. The rules are simple:

1. **Owner** — always has full access (read + write)
2. **Shared user with 'write'** — can read and write
3. **Shared user with 'read'** — can only read
4. **Everyone else** — no access

```
can_user_access(user_id, file_id, required_permission)

    ┌──────────────────┐
    │ Is user the owner?│
    └────────┬─────────┘
         YES │          NO
          ▼  │           ▼
     ✅ ALLOW    ┌────────────────────┐
                 │ Check shared_files  │
                 │ for (file, user)    │
                 └────────┬───────────┘
                     FOUND │       NOT FOUND
                       ▼   │           ▼
              ┌─────────────────┐   ❌ DENY
              │ permission >=   │
              │ required?       │
              └───┬─────────┬──┘
               YES│         │NO
                ▼ │         │ ▼
           ✅ ALLOW      ❌ DENY
```

Let's build this function and test it with our three users.

In [5]:
# Permission hierarchy: write includes read
PERMISSION_LEVELS = {"read": 1, "write": 2}


def can_user_access(user_id: int, file_id: int, required_permission: str) -> bool:
    """
    Check if a user can perform an action on a file.
    
    - Owner always has full access.
    - Shared users are checked against the shared_files table.
    - 'write' permission includes 'read'.
    """
    conn = get_db()
    cur = conn.cursor()
    
    # Step 1: Check if user is the owner
    cur.execute("SELECT owner_id FROM files WHERE id = %s", (file_id,))
    row = cur.fetchone()
    if row is None:
        conn.close()
        return False  # file doesn't exist
    
    if row[0] == user_id:
        conn.close()
        return True  # owner always has access
    
    # Step 2: Check shared_files for this (file, user)
    cur.execute("""
        SELECT permission FROM shared_files
        WHERE file_id = %s AND shared_with = %s
    """, (file_id, user_id))
    share = cur.fetchone()
    conn.close()
    
    if share is None:
        return False  # no share record — access denied
    
    # Step 3: Does their permission level meet the requirement?
    user_level = PERMISSION_LEVELS.get(share[0], 0)
    required_level = PERMISSION_LEVELS.get(required_permission, 0)
    return user_level >= required_level


# ── Test all combinations ────────────────────────────────────

users = {1: "alice (owner)", 2: "bob (read)", 3: "charlie (write)"}
permissions = ["read", "write"]

print(f"File: {file_name} (id={file_id})\n")
print(f"{'User':<22} {'Read?':<10} {'Write?':<10}")
print("-" * 42)
for uid, name in users.items():
    can_read = can_user_access(uid, file_id, "read")
    can_write = can_user_access(uid, file_id, "write")
    print(f"{name:<22} {'✅' if can_read else '❌':<10} {'✅' if can_write else '❌':<10}")

# Extra: a user with no share at all
print(f"\n{'dave (no share)':<22} {'✅' if can_user_access(999, file_id, 'read') else '❌':<10} {'✅' if can_user_access(999, file_id, 'write') else '❌':<10}")

File: project-plan.txt (id=24)

User                   Read?      Write?    
------------------------------------------
alice (owner)          ✅          ✅         
bob (read)             ✅          ❌         
charlie (write)        ✅          ✅         



dave (no share)        ❌          ❌         


## 🔒 Putting It Together: Secure Download

Now let's combine permission checks and presigned URLs into a realistic `generate_download_url` function. It checks permissions *first*, and only generates a URL if the user is allowed.

In [6]:
def generate_download_url(user_id: int, file_id: int, expires_in: int = 300) -> str:
    """
    Generate a presigned download URL for a file — but only if
    the user has at least 'read' permission.
    
    Returns the URL string, or raises PermissionError.
    """
    # Check permission
    if not can_user_access(user_id, file_id, "read"):
        raise PermissionError(f"User {user_id} cannot read file {file_id}")
    
    # Look up the storage key
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT storage_key FROM files WHERE id = %s", (file_id,))
    row = cur.fetchone()
    conn.close()
    
    if row is None:
        raise FileNotFoundError(f"File {file_id} not found")
    
    # Generate presigned URL
    s3 = get_s3()
    url = s3.generate_presigned_url(
        "get_object",
        Params={"Bucket": BUCKET, "Key": row[0]},
        ExpiresIn=expires_in
    )
    return url


# ── Demo: Bob can download, Dave cannot ──────────────────────

# Bob (read access) → should succeed
try:
    url = generate_download_url(user_id=2, file_id=file_id)
    print(f"✅ Bob got a download URL")
    print(f"   {url[:80]}...")
except PermissionError as e:
    print(f"❌ Bob denied: {e}")

# Unknown user (no access) → should fail
try:
    url = generate_download_url(user_id=999, file_id=file_id)
    print(f"\n✅ Unknown user got a URL (unexpected!)")
except PermissionError as e:
    print(f"\n❌ Unknown user denied: {e}")
    print("   This is correct — they have no share record.")

✅ Bob got a download URL
   http://localhost:9000/dropbox-files/alice/project-plan.txt?AWSAccessKeyId=minioa...

❌ Unknown user denied: User 999 cannot read file 24
   This is correct — they have no share record.


## 📂 Listing "Shared with Me" Efficiently

Every file-sharing app has a "Shared with me" view that shows all files others have shared with you. The **naive approach** would scan every file in the system and check if it's shared with you — that's slow!

Instead, we query the `shared_files` table directly. The index `idx_shared_user` on `shared_files(shared_with)` makes this fast even with millions of rows.

But even indexed queries have latency. If a user opens "Shared with me" repeatedly, we can **cache the result in Redis** with a short TTL. When a sharing change happens (new share or revoke), we **invalidate** the cache.

```
  User opens "Shared with me"
       │
       ▼
  ┌─────────────┐     HIT     ┌─────────┐
  │ Check Redis  │────────────►│ Return   │
  │ cache        │             │ cached   │
  └──────┬──────┘             └─────────┘
      MISS │
         ▼
  ┌──────────────┐
  │ Query Postgres│
  │ (indexed)     │
  └──────┬───────┘
         ▼
  ┌──────────────┐
  │ Store result  │
  │ in Redis      │
  │ (TTL = 60s)   │
  └──────┬───────┘
         ▼
  ┌─────────────┐
  │ Return to    │
  │ user         │
  └─────────────┘
```

In [7]:
SHARED_WITH_ME_TTL = 60  # cache for 60 seconds


def get_shared_with_me(user_id: int, use_cache: bool = True) -> list:
    """
    Return all files shared with a user.
    Uses Redis cache with TTL to avoid hitting Postgres every time.
    """
    r = get_redis()
    cache_key = f"shared_with_me:{user_id}"
    
    # Step 1: Check Redis cache
    if use_cache:
        cached = r.get(cache_key)
        if cached:
            print(f"   ⚡ Cache HIT for user {user_id}")
            return json.loads(cached)
        print(f"   💨 Cache MISS for user {user_id}")
    
    # Step 2: Query Postgres (uses idx_shared_user index)
    conn = get_db()
    cur = conn.cursor()
    cur.execute("""
        SELECT f.id, f.file_name, f.file_size, f.mime_type,
               sf.permission, u.username AS owner
        FROM shared_files sf
        JOIN files f ON f.id = sf.file_id
        JOIN users u ON u.id = f.owner_id
        WHERE sf.shared_with = %s
        ORDER BY sf.shared_at DESC
    """, (user_id,))
    
    columns = [desc[0] for desc in cur.description]
    results = [dict(zip(columns, row)) for row in cur.fetchall()]
    conn.close()
    
    # Step 3: Store in Redis with TTL
    r.setex(cache_key, SHARED_WITH_ME_TTL, json.dumps(results, default=str))
    print(f"   📝 Cached {len(results)} results (TTL={SHARED_WITH_ME_TTL}s)")
    
    return results


def invalidate_shared_cache(user_id: int):
    """Remove the cached 'shared with me' list when sharing changes."""
    r = get_redis()
    cache_key = f"shared_with_me:{user_id}"
    r.delete(cache_key)
    print(f"   🗑️  Invalidated cache for user {user_id}")


# ── Demo: First call = cache miss, second = cache hit ────────

print("📂 Bob's 'Shared with me' — first request:")
files = get_shared_with_me(user_id=2)
for f in files:
    print(f"   📄 {f['file_name']} ({f['permission']}) — owner: {f['owner']}")

print("\n📂 Bob's 'Shared with me' — second request:")
files = get_shared_with_me(user_id=2)
for f in files:
    print(f"   📄 {f['file_name']} ({f['permission']}) — owner: {f['owner']}")

print("\n🔍 Check RedisInsight — look for key 'shared_with_me:2'!")

📂 Bob's 'Shared with me' — first request:
   💨 Cache MISS for user 2
   📝 Cached 1 results (TTL=60s)
   📄 project-plan.txt (read) — owner: alice

📂 Bob's 'Shared with me' — second request:
   ⚡ Cache HIT for user 2
   📄 project-plan.txt (read) — owner: alice

🔍 Check RedisInsight — look for key 'shared_with_me:2'!


## ⏱️ Cache vs Database — Speed Comparison

Let's measure the difference between hitting Postgres and hitting the Redis cache. Even on a local machine, the cache is noticeably faster. In production (where the database might be on a different server), the gap is even bigger.

In [8]:
# Measure database query time (bypass cache)
db_times = []
for _ in range(50):
    start = time.time()
    get_shared_with_me(user_id=2, use_cache=False)
    db_times.append((time.time() - start) * 1000)

# Measure cache hit time (ensure cache is warm)
get_shared_with_me(user_id=2, use_cache=True)  # warm up
cache_times = []
for _ in range(50):
    start = time.time()
    get_shared_with_me(user_id=2, use_cache=True)
    cache_times.append((time.time() - start) * 1000)

avg_db = sum(db_times) / len(db_times)
avg_cache = sum(cache_times) / len(cache_times)

print("\n📊 Performance Comparison (50 iterations)")
print(f"   Database (Postgres):  avg {avg_db:.2f} ms")
print(f"   Cache (Redis):        avg {avg_cache:.2f} ms")
if avg_cache > 0:
    print(f"   Speedup:              {avg_db / avg_cache:.1f}×")

   📝 Cached 1 results (TTL=60s)


   📝 Cached 1 results (TTL=60s)
   📝 Cached 1 results (TTL=60s)
   📝 Cached 1 results (TTL=60s)


   📝 Cached 1 results (TTL=60s)


   📝 Cached 1 results (TTL=60s)
   📝 Cached 1 results (TTL=60s)
   📝 Cached 1 results (TTL=60s)


   📝 Cached 1 results (TTL=60s)


...

   📝 Cached 1 results (TTL=60s)
   📝 Cached 1 results (TTL=60s)


   📝 Cached 1 results (TTL=60s)


   📝 Cached 1 results (TTL=60s)
   📝 Cached 1 results (TTL=60s)
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2


   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2


   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2
   ⚡ Cache HIT for user 2

📊 Performance Comparison (50 iterations)
   Database (Postgres):  avg 24.82 ms
   Cache (Redis):        avg 2.49 ms
   Speedup:              10.0×


## 🚫 Revoking Access

Sometimes you need to **un-share** a file. Maybe a teammate left the project, or you shared with the wrong person. Revoking is simple: delete the row from `shared_files`.

But there's a **gotcha**: presigned URLs that were already generated *still work* until they expire. This is the "bearer token" nature — the URL contains everything needed to download the file, and S3 doesn't check our database.

```
  Timeline:
  ─────────────────────────────────────────────────►
  │                │                    │
  Share file       Revoke access        URL expires
  + generate URL   (delete from DB)     (S3 rejects)
  │                │                    │
  │◄── URL works ──►◄── URL STILL works ►│
  │                │   (gap of risk)     │
```

**Mitigation**: keep presigned URL TTLs short (e.g., 5 minutes instead of 24 hours). The shorter the TTL, the smaller the window where a revoked user can still download.

In [9]:
# ── Step 1: Generate a URL for Bob BEFORE revoking ───────────
s3 = get_s3()
bobs_url = s3.generate_presigned_url(
    "get_object",
    Params={"Bucket": BUCKET, "Key": storage_key},
    ExpiresIn=10  # 10 seconds for demo
)
print("🔗 Generated a presigned URL for Bob (expires in 10s)")

# ── Step 2: Revoke Bob's access ──────────────────────────────
conn = get_db()
cur = conn.cursor()
cur.execute("""
    DELETE FROM shared_files
    WHERE file_id = %s AND shared_with = 2
    RETURNING id
""", (file_id,))
deleted = cur.fetchone()
conn.close()

print(f"🚫 Revoked Bob's access (deleted share id={deleted[0]})")

# Invalidate Bob's cache
invalidate_shared_cache(user_id=2)

# ── Step 3: Check — Bob's old URL still works! ───────────────
response = requests.get(bobs_url)
print(f"\n⬇️  Bob's old URL after revoke: HTTP {response.status_code}")
if response.status_code == 200:
    print("   ⚠️  URL still works! It was already signed.")
    print("   This is why short TTLs are important.")

# ── Step 4: But Bob can't get a NEW URL ──────────────────────
try:
    url = generate_download_url(user_id=2, file_id=file_id)
    print(f"\n✅ Bob got a new URL (unexpected!)")
except PermissionError as e:
    print(f"\n❌ Bob can't get a new URL: {e}")
    print("   Access properly revoked — no new URLs will be generated.")

# ── Step 5: Verify permission check reflects the revoke ──────
print(f"\n📋 Bob's access after revoke:")
print(f"   Read:  {'✅' if can_user_access(2, file_id, 'read') else '❌'}")
print(f"   Write: {'✅' if can_user_access(2, file_id, 'write') else '❌'}")

🔗 Generated a presigned URL for Bob (expires in 10s)


🚫 Revoked Bob's access (deleted share id=7)
   🗑️  Invalidated cache for user 2

⬇️  Bob's old URL after revoke: HTTP 200
   ⚠️  URL still works! It was already signed.
   This is why short TTLs are important.

❌ Bob can't get a new URL: User 2 cannot read file 24
   Access properly revoked — no new URLs will be generated.

📋 Bob's access after revoke:
   Read:  ❌
   Write: ❌


## 🔄 Cache Invalidation in Action

Let's see the full cycle: share → cache → revoke → invalidate → re-query. This demonstrates how the cache stays in sync with the database.

In [10]:
# Re-share with Bob so we can demonstrate the full cycle
conn = get_db()
cur = conn.cursor()
cur.execute("""
    INSERT INTO shared_files (file_id, shared_with, permission)
    VALUES (%s, 2, 'read')
    ON CONFLICT (file_id, shared_with) DO UPDATE SET permission = 'read'
""", (file_id,))
conn.close()
invalidate_shared_cache(user_id=2)
print("✅ Re-shared file with Bob\n")

# 1. First query — cache miss, loads from DB
print("Step 1: Query 'Shared with me' (cache miss)")
files = get_shared_with_me(user_id=2)
print(f"   Files: {[f['file_name'] for f in files]}\n")

# 2. Second query — cache hit
print("Step 2: Query again (cache hit)")
files = get_shared_with_me(user_id=2)
print(f"   Files: {[f['file_name'] for f in files]}\n")

# 3. Revoke access → invalidate cache
print("Step 3: Revoke Bob's access + invalidate cache")
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM shared_files WHERE file_id = %s AND shared_with = 2", (file_id,))
conn.close()
invalidate_shared_cache(user_id=2)
print()

# 4. Query again — cache miss, DB returns empty
print("Step 4: Query after revoke (cache miss → empty list)")
files = get_shared_with_me(user_id=2)
print(f"   Files: {[f['file_name'] for f in files]}")
print("\n✅ Cache stays in sync with the database!")

   🗑️  Invalidated cache for user 2
✅ Re-shared file with Bob

Step 1: Query 'Shared with me' (cache miss)
   💨 Cache MISS for user 2


   📝 Cached 1 results (TTL=60s)
   Files: ['project-plan.txt']

Step 2: Query again (cache hit)
   ⚡ Cache HIT for user 2
   Files: ['project-plan.txt']

Step 3: Revoke Bob's access + invalidate cache
   🗑️  Invalidated cache for user 2

Step 4: Query after revoke (cache miss → empty list)


   💨 Cache MISS for user 2


   📝 Cached 0 results (TTL=60s)
   Files: []

✅ Cache stays in sync with the database!


## 🔔 Sharing Notifications via Sync Events

When Alice shares a file with Bob, Bob needs to know about it. In a real system, you might use push notifications or WebSockets. In our simplified model, we use the `sync_events` table that clients **poll** periodically.

When a file is shared, we insert a sync event of type `'shared'` for the recipient. The recipient's app checks for new events since the last one it saw.

```
  Alice shares file          Bob's app polls sync_events
       │                              │
       ▼                              ▼
  INSERT INTO shared_files    SELECT * FROM sync_events
  INSERT INTO sync_events     WHERE user_id = bob
    (type='shared',             AND id > last_seen_id
     user_id=bob)             → "Alice shared project-plan.txt"
```

Let's share the file with Bob again and see how the notification works.

In [11]:
def share_file(file_id: int, owner_id: int, target_user_id: int, permission: str):
    """
    Share a file with a user and create a notification.
    This combines:
      1. Insert into shared_files
      2. Insert a sync_event for the recipient
      3. Invalidate the recipient's cache
    """
    conn = get_db()
    cur = conn.cursor()
    
    # 1. Create or update the share
    cur.execute("""
        INSERT INTO shared_files (file_id, shared_with, permission)
        VALUES (%s, %s, %s)
        ON CONFLICT (file_id, shared_with)
        DO UPDATE SET permission = EXCLUDED.permission
        RETURNING id
    """, (file_id, target_user_id, permission))
    share_id = cur.fetchone()[0]
    
    # 2. Create a sync event so the recipient discovers the share
    cur.execute("""
        INSERT INTO sync_events (user_id, file_id, event_type)
        VALUES (%s, %s, 'shared')
        RETURNING id
    """, (target_user_id, file_id))
    event_id = cur.fetchone()[0]
    
    conn.close()
    
    # 3. Invalidate the recipient's "shared with me" cache
    invalidate_shared_cache(target_user_id)
    
    return share_id, event_id


# ── Alice shares the file with Bob ───────────────────────────
share_id, event_id = share_file(
    file_id=file_id,
    owner_id=1,       # Alice
    target_user_id=2, # Bob
    permission="read"
)
print(f"✅ Alice shared file with Bob — share id={share_id}, event id={event_id}")

   🗑️  Invalidated cache for user 2
✅ Alice shared file with Bob — share id=10, event id=26


In [12]:
def poll_notifications(user_id: int, last_seen_event_id: int = 0) -> list:
    """
    Simulate a client polling for new sync events.
    Returns events newer than last_seen_event_id.
    """
    conn = get_db()
    cur = conn.cursor()
    cur.execute("""
        SELECT se.id, se.event_type, se.created_at,
               f.file_name, u.username AS owner
        FROM sync_events se
        JOIN files f ON f.id = se.file_id
        JOIN users u ON u.id = f.owner_id
        WHERE se.user_id = %s AND se.id > %s
        ORDER BY se.id ASC
    """, (user_id, last_seen_event_id))
    
    columns = [desc[0] for desc in cur.description]
    events = [dict(zip(columns, row)) for row in cur.fetchall()]
    conn.close()
    return events


# ── Bob checks for new notifications ─────────────────────────

print("📱 Bob's app polls for new events...")
events = poll_notifications(user_id=2, last_seen_event_id=0)

if events:
    print(f"   🔔 {len(events)} new notification(s):\n")
    for e in events:
        print(f"   Event #{e['id']}: {e['event_type']}")
        print(f"     File: {e['file_name']} (owner: {e['owner']})")
        print(f"     Time: {e['created_at']}")
        print()
    
    # Bob remembers the last event ID for next poll
    last_seen = events[-1]["id"]
    print(f"   Bob saves last_seen_event_id = {last_seen}")
    print(f"   Next poll will only return events after #{last_seen}")
else:
    print("   No new notifications.")

# ── Poll again — nothing new ─────────────────────────────────
print("\n📱 Bob polls again immediately...")
new_events = poll_notifications(user_id=2, last_seen_event_id=last_seen)
print(f"   {len(new_events)} new notification(s) — nothing new!")

📱 Bob's app polls for new events...
   🔔 1 new notification(s):

   Event #26: shared
     File: project-plan.txt (owner: alice)
     Time: 2026-04-19 21:26:02.201706

   Bob saves last_seen_event_id = 26
   Next poll will only return events after #26

📱 Bob polls again immediately...
   0 new notification(s) — nothing new!


## 🧩 Putting It All Together

Let's walk through the complete sharing flow one more time, from start to finish, to see how all the pieces connect:

```
  1. Alice uploads a file           → files table + MinIO
  2. Alice shares with Bob (read)   → shared_files table
  3. Notification sent              → sync_events table
  4. Bob's cache invalidated        → Redis DELETE
  5. Bob polls notifications        → sees "shared" event
  6. Bob opens "Shared with me"     → Redis cache (or Postgres)
  7. Bob requests download          → permission check → presigned URL
  8. Bob downloads from S3          → direct from MinIO
  9. Alice revokes access           → DELETE shared_files + invalidate cache
  10. Bob can't get new URLs        → permission check fails
```

### 💡 Key Design Decisions

| Decision | Why |
|----------|-----|
| Simple read/write permissions | Easy to understand and implement; covers 90% of use cases |
| Presigned URLs instead of proxying | Offloads bandwidth to S3; our server just does auth |
| Short TTL on presigned URLs | Limits damage if a URL leaks or access is revoked |
| Redis cache for "Shared with me" | Frequently accessed, rarely changes — perfect for caching |
| Cache invalidation on share/revoke | Ensures consistency when permissions change |
| Sync events for notifications | Simple polling model; easy to upgrade to WebSockets later |

## 🧹 Cleanup

In [13]:
# Clean up all data created in this notebook
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM sync_events")
cur.execute("DELETE FROM shared_files")
cur.execute("DELETE FROM chunks")
cur.execute("DELETE FROM files")
conn.close()
print("🧹 Cleaned up Postgres tables")

# Clean up Redis keys
r = get_redis()
keys = r.keys("shared_with_me:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 No Redis keys to clean up")

# Clean up MinIO objects
s3 = get_s3()
try:
    objects = s3.list_objects_v2(Bucket=BUCKET).get("Contents", [])
    for obj in objects:
        s3.delete_object(Bucket=BUCKET, Key=obj["Key"])
    print(f"🧹 Cleaned up {len(objects)} MinIO objects")
except Exception:
    print("🧹 No MinIO objects to clean up")

print("\n✅ All clean!")

🧹 Cleaned up Postgres tables
🧹 No Redis keys to clean up
🧹 Cleaned up 1 MinIO objects

✅ All clean!


## 📚 Summary

### Key Takeaways

1. **`shared_files` is the source of truth** — every share is a row with `(file_id, user_id, permission)`
2. **Presigned URLs are time-limited bearer tokens** — they give direct S3 access without proxying through your server
3. **Permission checks happen at your API layer** — S3 doesn't know about your users; you check first, then generate the URL
4. **Revoking access has a gap** — already-generated URLs work until expiry; use short TTLs to minimize risk
5. **Cache "Shared with me" in Redis** — it's read-heavy and changes infrequently; invalidate on share/revoke
6. **Sync events enable notifications** — a simple polling model that can be upgraded to WebSockets later

---

### 🎉 Congratulations — You Completed the Dropbox System Design Series!

Across four notebooks, you've built the core pieces of a Dropbox-like system:

| Notebook | What You Built |
|----------|----------------|
| **01 — Chunked File Uploads** | Chunked, resumable uploads with presigned URLs and integrity checks |
| **02 — File Sync & Conflict Resolution** | Event log, polling + Redis pub/sub, optimistic versioning, keep-both conflicts |
| **03 — Deduplication** | File- and chunk-level dedup, content-defined chunking, reference counting |
| **04 — Sharing & Permissions** | Access control, presigned URLs, caching, notifications |

These are the same building blocks that power real cloud storage systems. Keep experimenting! 🚀